# PyTorch to Core ML Conversion for Pupl App

This notebook converts your PyTorch pupil size prediction models to Core ML format using Google Colab's environment to bypass local AVX issues.

## Setup Instructions
1. Upload your PyTorch models (`left_eye_mobile.pt`, `right_eye_mobile.pt`) to this Colab
2. Upload your Firebase service account key JSON file
3. Run all cells
4. Download the converted `.mlmodel` files

---

## 1. Install Required Packages

In [ ]:
# Install required packages
!pip install coremltools>=6.0
!pip install torch torchvision
!pip install firebase-admin
!pip install onnx>=1.12.0
!pip install onnxruntime>=1.12.0

print("✅ All packages installed successfully!")

## 2. Import Libraries

In [ ]:
import torch
import torchvision
import coremltools as ct
import numpy as np
import os
import json
import firebase_admin
from firebase_admin import credentials, storage
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CoreMLTools version: {ct.__version__}")
print(f"TensorFlow will not be used - bypassing AVX issues!")

## 3. Upload PyTorch Models

Upload your PyTorch model files here:

In [ ]:
# Upload PyTorch models
print("Please upload your PyTorch model files:")
print("- left_eye_mobile.pt")
print("- right_eye_mobile.pt")

uploaded = files.upload()

# Check uploaded files
model_files = []
for filename in uploaded.keys():
    if filename.endswith('.pt'):
        model_files.append(filename)
        print(f"✅ Found model: {filename}")

if len(model_files) == 0:
    print("❌ No .pt files found. Please upload your PyTorch models.")
else:
    print(f"📦 Ready to convert {len(model_files)} models")

## 4. Firebase Setup (Optional)

Upload your Firebase service account key to automatically upload converted models:

In [ ]:
# Optional: Firebase setup for automatic upload
firebase_enabled = False

try:
    print("Upload your Firebase service account key JSON file (optional):")
    firebase_key = files.upload()
    
    if firebase_key:
        key_filename = list(firebase_key.keys())[0]
        
        # Initialize Firebase
        cred = credentials.Certificate(key_filename)
        firebase_admin.initialize_app(cred, {
            'storageBucket': 'your-project-id.appspot.com'  # Replace with your project ID
        })
        
        firebase_enabled = True
        print("✅ Firebase initialized successfully!")
        
except Exception as e:
    print(f"⚠️ Firebase setup skipped: {e}")
    print("Models will be available for manual download.")

## 5. Conversion Functions

In [ ]:
def test_pytorch_model(model_path):
    """Test if PyTorch model loads and runs correctly"""
    
    try:
        print(f"🔍 Testing {model_path}...")
        
        # Load model
        model = torch.jit.load(model_path, map_location='cpu')
        model.eval()
        
        # Test with dummy input (32x64 RGB image)
        dummy_input = torch.rand(1, 3, 32, 64)
        
        with torch.no_grad():
            output = model(dummy_input)
            print(f"  ✅ Model test passed: {output.item():.3f}")
            print(f"  📊 Input shape: {dummy_input.shape}")
            print(f"  📊 Output shape: {output.shape}")
            
        return True, model, dummy_input
        
    except Exception as e:
        print(f"  ❌ Model test failed: {e}")
        return False, None, None

def convert_to_coreml(model, dummy_input, output_name):
    """Convert PyTorch model to Core ML"""
    
    try:
        print(f"🔄 Converting {output_name}...")
        
        # Convert to Core ML
        coreml_model = ct.convert(
            model,
            inputs=[
                ct.ImageType(
                    name="input_image",
                    shape=(1, 3, 32, 64),
                    scale=1.0/255.0,
                    color_layout=ct.colorlayout.RGB
                )
            ],
            outputs=[
                ct.TensorType(name="pupil_diameter")
            ],
            minimum_deployment_target=ct.target.iOS13,
            compute_units=ct.ComputeUnit.ALL  # Use all available compute units
        )
        
        # Add metadata
        coreml_model.short_description = f"Pupil diameter prediction for {output_name.replace('_', ' ')}"
        coreml_model.author = "Pupl Team"
        coreml_model.version = "1.0"
        coreml_model.license = "Proprietary"
        
        # Save model
        output_path = f"{output_name}.mlmodel"
        coreml_model.save(output_path)
        
        # Get file size
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"  ✅ Converted to {output_path} ({size_mb:.1f} MB)")
        
        return True, output_path
        
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        import traceback
        traceback.print_exc()
        return False, None

def upload_to_firebase(file_path):
    """Upload converted model to Firebase Storage"""
    
    if not firebase_enabled:
        return False
        
    try:
        bucket = storage.bucket()
        blob = bucket.blob(f"models/{os.path.basename(file_path)}")
        blob.upload_from_filename(file_path)
        
        print(f"  ☁️ Uploaded to Firebase: {blob.public_url}")
        return True
        
    except Exception as e:
        print(f"  ❌ Firebase upload failed: {e}")
        return False

print("✅ Conversion functions defined successfully!")

## 6. Main Conversion Process

In [ ]:
# Main conversion process
print("🚀 Starting PyTorch to Core ML conversion...")
print("=" * 60)

conversion_results = []
successful_conversions = 0

# Define model mappings
model_mappings = {
    'left_eye_mobile.pt': 'left_eye',
    'right_eye_mobile.pt': 'right_eye',
    'left_eye.pt': 'left_eye_full',
    'right_eye.pt': 'right_eye_full'
}

# Process each uploaded model
for model_file in model_files:
    print(f"\n📦 Processing {model_file}:")
    print("-" * 40)
    
    # Determine output name
    output_name = model_mappings.get(model_file, model_file.replace('.pt', ''))
    
    # Test PyTorch model
    success, model, dummy_input = test_pytorch_model(model_file)
    
    if not success:
        conversion_results.append({
            'input': model_file,
            'output': None,
            'success': False,
            'error': 'PyTorch model test failed'
        })
        continue
    
    # Convert to Core ML
    success, output_path = convert_to_coreml(model, dummy_input, output_name)
    
    if not success:
        conversion_results.append({
            'input': model_file,
            'output': None,
            'success': False,
            'error': 'Core ML conversion failed'
        })
        continue
    
    # Upload to Firebase (optional)
    firebase_success = upload_to_firebase(output_path)
    
    # Record success
    conversion_results.append({
        'input': model_file,
        'output': output_path,
        'success': True,
        'firebase_uploaded': firebase_success
    })
    
    successful_conversions += 1
    print(f"  🎉 Successfully converted {model_file} → {output_path}")

print(f"\n{'='*60}")
print(f"🏁 Conversion completed!")
print(f"✅ Successfully converted: {successful_conversions}/{len(model_files)} models")

# Summary
if successful_conversions > 0:
    print(f"\n📊 Conversion Summary:")
    for result in conversion_results:
        if result['success']:
            size_mb = os.path.getsize(result['output']) / (1024 * 1024)
            print(f"  ✅ {result['input']} → {result['output']} ({size_mb:.1f} MB)")
        else:
            print(f"  ❌ {result['input']} → {result['error']}")
else:
    print("\n❌ No models were converted successfully")
    print("Please check the error messages above and try again.")

## 7. Download Converted Models

In [ ]:
# Download converted models
print("📥 Downloading converted Core ML models...")

converted_files = []
for result in conversion_results:
    if result['success'] and result['output']:
        converted_files.append(result['output'])

if converted_files:
    print(f"Found {len(converted_files)} converted models:")
    for file in converted_files:
        print(f"  - {file}")
    
    # Download all converted models
    for file in converted_files:
        if os.path.exists(file):
            files.download(file)
            print(f"✅ Downloaded: {file}")
        else:
            print(f"❌ File not found: {file}")
else:
    print("❌ No converted models available for download")

print("\n🎉 All done! Add the downloaded .mlmodel files to your Xcode project.")

## 8. Integration Instructions

### Next Steps:

1. **Download the .mlmodel files** from the cell above
2. **Add to Xcode project:**
   - Drag the `.mlmodel` files into your Xcode project
   - Make sure "Add to target" is checked for PupillometryApp
   - Ensure "Copy items if needed" is selected

3. **Remove placeholder models:**
   ```bash
   cd /path/to/your/project
   rm left_eye.mlmodel right_eye.mlmodel  # Remove old placeholder files
   ```

4. **Build and test:**
   - Build your iOS app
   - Look for "Core ML inference" in the logs
   - Your app should now use the real trained models!

### Expected Behavior:
- App loads Core ML models successfully
- Real PyTorch model inference provides trained accuracy
- MediaPipe landmarks feed precise eye regions to models
- Pupil diameter predictions use your trained neural networks

### Troubleshooting:
- If models don't load, check Xcode console for errors
- App will fall back to MediaPipe estimation if Core ML fails
- MediaPipe estimation is already highly accurate as backup

---

**Your pupillometry app now has real PyTorch model inference! 🎉**